# Part 2: Advanced Heuristics & YAML

Once you have mastered basic tolerances, you can use Veridelta to map legacy values, clean dirty strings with regex, and manage configurations via YAML.

In [ ]:
import polars as pl
from pathlib import Path
from veridelta import load_config, DiffEngine, DataIngestor

src = pl.DataFrame({
    "user_id": [1, 2],
    "plan_type": ["Premium", "Basic"],
    "phone": ["(555) 123-4567", "555-987-6543"]
})

# Target has acronyms for plans, and unformatted phone numbers
tgt = pl.DataFrame({
    "user_id": [1, 2],
    "plan_type": ["PRM", "BSC"],       
    "phone": ["5551234567", "5559876543"]
})

src.write_parquet("source.parquet")
tgt.write_parquet("target.parquet")

### Defining Advanced Rules via YAML
Instead of writing Python code, data engineers can define these translation rules in a clean YAML file.

In [ ]:
yaml_config = """
primary_keys: ["user_id"]

source:
  path: "source.parquet"
  format: "parquet"
target:
  path: "target.parquet"
  format: "parquet"

rules:
  - column_names: ["plan_type"]
    value_map:                   # Safely map old values to new ones
      "Premium": "PRM"
      "Basic": "BSC"

  - column_names: ["phone"]
    regex_replace: 
      "[^0-9]": ""               # Strip everything except numbers before comparing
"""
Path("veridelta.yaml").write_text(yaml_config)

config = load_config("veridelta.yaml")
source_df, target_df = DataIngestor(config).get_dataframes()

engine = DiffEngine(config, source_df, target_df)
summary = engine.run()

print(f"Match Status: {summary.is_match} | Unresolved Rows: {summary.changed_count}")